# Building Agentic Workflows with LlamaIndex

**Goal:** Build a task-oriented agent using LlamaIndex components:
- QueryEngine (RAG)
- Tools (calculator, URL summarizer stub)
- Router (choose tool vs RAG vs plain LLM)
- Memory (conversation + vector memory)
- Simple policies (allowlist, input guards)

**Estimated Time:** ~120 minutes

## 1) Setup

In [ ]:
%pip install llama-index-core llama-index-embeddings-openai llama-index-llms-openai python-dotenv requests beautifulsoup4

Create a `.env` file in this folder with:

```env
OPENAI_API_KEY=your_api_key_here
```

## 2) Initialize LLM, embeddings, and index

In [ ]:
import os, re, ast, json, operator as op, requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, ChatPromptTemplate
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it to your .env file.")

llm = OpenAI(model="gpt-4o-mini", temperature=0)
emb = OpenAIEmbedding(model="text-embedding-3-small")

# Reuse data from Lab 1 or create quickly
os.makedirs("data", exist_ok=True)
seed = {
  "rag.txt": "RAG grounds answers by retrieving relevant documents before generation.",
  "agents.txt": "Agentic AI uses memory, tools, and goals to act autonomously.",
  "langgraph.txt": "LangGraph provides explicit nodes and edges for stateful workflows."
}
for n, t in seed.items():
    with open(f"data/{n}", "w", encoding="utf-8") as f:
        f.write(t)

docs = SimpleDirectoryReader("data").load_data()
index = VectorStoreIndex.from_documents(docs, embed_model=emb)
qe = index.as_query_engine(llm=llm)

print(f"Loaded {len(docs)} docs and built RAG index.")

## 3) Add long-term memory (appendable)

In [ ]:
mem_index = VectorStoreIndex.from_documents(
    [Document(text="User prefers concise answers.")],
    embed_model=emb
)
mem_qe = mem_index.as_query_engine(llm=llm)

def save_memory(text: str):
    mem_index.insert(Document(text=text))

## 4) Tools

In [ ]:
# Safe calculator
OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
    ast.Mod: op.mod,
}

def safe_calc(expr: str) -> str:
    node = ast.parse(expr, mode="eval").body

    def eval_(x):
        if isinstance(x, ast.Num):
            return x.n
        if isinstance(x, ast.UnaryOp) and type(x.op) in OPS:
            return OPS[type(x.op)](eval_(x.operand))
        if isinstance(x, ast.BinOp) and type(x.op) in OPS:
            return OPS[type(x.op)](eval_(x.left), eval_(x.right))
        raise ValueError("disallowed")

    return str(eval_(node))

# Allowlisted URL summarizer (simple)
ALLOWLIST = {"python.org", "openai.com"}

def fetch_allowlisted(url: str, limit=4000):
    m = re.search(r'(https?://\S+)', url)
    if not m:
        return "Invalid URL."

    target = m.group(1)
    host = re.sub(r"^https?://", "", target).split("/")[0].lower()
    if not any(host.endswith(h) for h in ALLOWLIST):
        return "Blocked: domain not allowlisted."

    r = requests.get(target, timeout=12, headers={"User-Agent": "LlamaIndex-Agent/1.0"})
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    text = " ".join(p.get_text(" ", strip=True) for p in soup.find_all("p"))[:limit]
    return text if text else "No readable content."

## 5) Router (LLM decides route)

In [ ]:
ROUTER_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a router. Given a user query, choose exactly one route: CALC, URL, RAG, or LLM."),
        ("user", """Query: {query}\nDecide route with terse justification. Reply as JSON: {\"route\": \"...\", \"expr_or_url\": \"\"}"""),
    ]
)

def route(query: str):
    msg = llm.chat(ROUTER_PROMPT.format_messages(query=query)).message.content

    try:
        obj = json.loads(msg)
    except Exception:
        obj = {"route": "RAG", "expr_or_url": ""}

    r = str(obj.get("route", "RAG")).upper()
    val = str(obj.get("expr_or_url", "")).strip()

    if r not in {"CALC", "URL", "RAG", "LLM"}:
        r = "RAG"

    return r, val

## 6) Orchestrator

In [ ]:
chat_history = []  # short-term memory

BLOCK_PATTERNS = [
    r"(?i)ignore previous",
    r"(?i)disable safety",
    r"(?i)reveal secret",
    r"(?i)exfiltrate",
]

def guard(text: str):
    for p in BLOCK_PATTERNS:
        if re.search(p, text):
            raise PermissionError("Blocked by safety policy.")

SYSTEM_GUIDE = (
    "You are a helpful agent. Be concise. "
    "Prefer RAG for conceptual questions, CALC for arithmetic, URL for summaries of allowlisted pages. "
    "Use long-term memory notes when relevant."
)

def answer(query: str):
    guard(query)

    # 1) route
    route_name, aux = route(query)

    # 2) recall long-term memory
    mem_hits = mem_qe.query(f"What should I remember relevant to: {query}?").response

    # 3) execute selected path
    if route_name == "CALC":
        out = safe_calc(aux or query)
    elif route_name == "URL":
        raw = fetch_allowlisted(aux or query)
        out = llm.complete(f"Summarize in <=120 words:\n{raw}").text
    elif route_name == "RAG":
        out = qe.query(f"{query}\n(Consider: {mem_hits})").response
    else:
        out = llm.complete(f"{SYSTEM_GUIDE}\nHistory:{chat_history[-4:]}\nUser:{query}").text

    # 4) conversational memory update
    chat_history.append({"role": "user", "content": query})
    chat_history.append({"role": "assistant", "content": out})
    if len(chat_history) > 12:
        del chat_history[:len(chat_history)-12]

    # 5) heuristic memory save
    if re.search(r"(?i)\bremember that\b", query):
        save_memory(query)

    return out

## 7) Try it out

In [ ]:
print("Q1:", "What is RAG and why is it useful?")
print("A1:", answer("What is RAG and why is it useful?"))

print("\nQ2:", "Compute 15*(4+2)")
print("A2:", answer("Compute 15*(4+2)"))

print("\nQ3:", "Summarize https://python.org in 2 sentences")
print("A3:", answer("Summarize https://python.org in 2 sentences"))

print("\nQ4:", "Remember that I prefer bullet-point answers.")
print("A4:", answer("Remember that I prefer bullet-point answers."))

print("\nQ5:", "Explain LangGraph briefly, adapt to my preference.")
print("A5:", answer("Explain LangGraph briefly, adapt to my preference."))

## 8) Interactive Query Loop (optional)

In [ ]:
while True:
    q = input("Ask a question (or 'exit'): " ).strip()
    if q.lower() in {"exit", "quit"}:
        print("Goodbye!")
        break
    try:
        print("A:", answer(q))
    except Exception as e:
        print("Error:", e)

## 9) Optional Upgrades
- Function-calling tools: wrap tools as LlamaIndex Tool objects and use AgentRunner
- Response synthesis: mix multiple indices (e.g., ComposableGraph)
- Memory policies: whitelist remember statements, summarize memories nightly
- Structured outputs: enforce JSON responses for integrations
- Telemetry: log route decisions + latency

## 10) Troubleshooting
- If router misroutes, add few-shot examples to `ROUTER_PROMPT`
- If URL is blocked, extend `ALLOWLIST`
- If retrieval is weak, increase `similarity_top_k` and enrich `data/` docs

You now have a LlamaIndex-driven agentic workflow:
**router → (RAG / Tool / LLM) → memory → response**